# Phase 1 — Dataset Preparation
### SQL Fine-Tuning Project | Spider Dataset → Qwen 1.5B Instruction Format

This notebook:
1. Loads the Spider Text-to-SQL dataset from Hugging Face
2. Formats it into instruction-response pairs
3. Splits into train/validation sets
4. Saves everything to disk


In [ ]:
import os
import json
import pandas as pd
from datasets import load_dataset, DatasetDict
from pathlib import Path

print("All imports successful!")


## Step 1 — Configuration

In [ ]:
# ── Project Configuration ──────────────────────────────────────────
CONFIG = {
    "dataset_name"    : "xlangai/spider",   # Spider dataset on HuggingFace
    "model_name"      : "Qwen/Qwen1.5-1.8B-Chat",
    "output_dir"      : "./data",           # Where to save processed data
    "train_split"     : 0.9,               # 90% train, 10% validation
    "max_samples"     : None,              # None = use full dataset
    "seed"            : 42,
}

# Create output directory
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
print("Config ready!")
print(f"Dataset  : {CONFIG['dataset_name']}")
print(f"Model    : {CONFIG['model_name']}")
print(f"Save dir : {CONFIG['output_dir']}")


## Step 2 — Load Spider Dataset

In [ ]:
# Load Spider dataset from Hugging Face
print("Downloading Spider dataset...")
print("This may take a minute on first run...\n")

dataset = load_dataset(CONFIG["dataset_name"])

print("Dataset loaded successfully!")
print(f"\nDataset structure:")
print(dataset)
print(f"\nTrain samples : {len(dataset['train'])}")
print(f"Valid samples  : {len(dataset['validation'])}")


## Step 3 — Explore the Data

In [ ]:
# Look at a sample entry
sample = dataset['train'][0]
print("Sample entry keys:", list(sample.keys()))
print("\n" + "="*60)
print("QUESTION  :", sample['question'])
print("QUERY     :", sample['query'])
print("DB ID     :", sample['db_id'])
print("="*60)

# Show a few more examples
print("\n--- More examples ---")
for i in [1, 2, 3]:
    s = dataset['train'][i]
    print(f"\nExample {i+1}:")
    print(f"  Q: {s['question']}")
    print(f"  SQL: {s['query']}")


## Step 4 — Format into Instruction-Response Pairs

In [ ]:
def format_instruction(sample):
    """
    Convert a Spider sample into Qwen chat instruction format.
    
    Input  : Natural language question + database id
    Output : Formatted SQL query
    """
    instruction = f"""You are an expert SQL assistant. Given a natural language question and a database name, write the correct SQL query.

Database: {sample['db_id']}
Question: {sample['question']}

Write only the SQL query, nothing else."""

    response = sample['query'].strip()

    # Qwen chat format
    formatted = f"""<|im_start|>system
You are an expert SQL assistant that converts natural language questions into accurate SQL queries.<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""

    return {
        "instruction" : instruction,
        "response"    : response,
        "text"        : formatted,
        "db_id"       : sample['db_id'],
        "question"    : sample['question'],
        "query"       : sample['query'],
    }

# Test the formatter on one sample
test = format_instruction(dataset['train'][0])
print("Formatted sample:")
print("="*60)
print(test['text'])
print("="*60)


In [ ]:
# Apply formatting to entire dataset
print("Formatting dataset...")

train_formatted = dataset['train'].map(
    format_instruction,
    remove_columns=dataset['train'].column_names,
    desc="Formatting train set"
)

val_formatted = dataset['validation'].map(
    format_instruction,
    remove_columns=dataset['validation'].column_names,
    desc="Formatting validation set"
)

print(f"\nFormatting complete!")
print(f"Train samples     : {len(train_formatted)}")
print(f"Validation samples: {len(val_formatted)}")


## Step 5 — Dataset Statistics

In [ ]:
# Analyze text lengths
train_lengths = [len(x['text'].split()) for x in train_formatted]
val_lengths   = [len(x['text'].split()) for x in val_formatted]

print("=== Dataset Statistics ===")
print(f"\nTrain set:")
print(f"  Total samples : {len(train_formatted)}")
print(f"  Avg length    : {sum(train_lengths)/len(train_lengths):.0f} tokens")
print(f"  Max length    : {max(train_lengths)} tokens")
print(f"  Min length    : {min(train_lengths)} tokens")

print(f"\nValidation set:")
print(f"  Total samples : {len(val_formatted)}")
print(f"  Avg length    : {sum(val_lengths)/len(val_lengths):.0f} tokens")
print(f"  Max length    : {max(val_lengths)} tokens")
print(f"  Min length    : {min(val_lengths)} tokens")

# Show unique databases
unique_dbs = set(train_formatted['db_id'])
print(f"\nUnique databases in train: {len(unique_dbs)}")
print(f"Sample DBs: {list(unique_dbs)[:8]}")


## Step 6 — Save Processed Dataset

In [ ]:
# Save as HuggingFace DatasetDict
final_dataset = DatasetDict({
    "train"     : train_formatted,
    "validation": val_formatted,
})

# Save to disk
save_path = CONFIG["output_dir"] + "/spider_formatted"
final_dataset.save_to_disk(save_path)
print(f"Dataset saved to: {save_path}")

# Also save as JSON for easy inspection
train_formatted.to_json(CONFIG["output_dir"] + "/train.json")
val_formatted.to_json(CONFIG["output_dir"] + "/validation.json")
print(f"JSON files saved to: {CONFIG['output_dir']}/")

# Save config
with open(CONFIG["output_dir"] + "/dataset_config.json", "w") as f:
    json.dump({
        "dataset_name"       : CONFIG["dataset_name"],
        "train_samples"      : len(train_formatted),
        "validation_samples" : len(val_formatted),
        "format"             : "qwen_chat",
        "model_target"       : CONFIG["model_name"],
    }, f, indent=2)

print("\n=== All files saved! ===")
print(f"  {save_path}/")
print(f"  {CONFIG['output_dir']}/train.json")
print(f"  {CONFIG['output_dir']}/validation.json")
print(f"  {CONFIG['output_dir']}/dataset_config.json")


## Step 7 — Verify Saved Dataset

In [ ]:
from datasets import load_from_disk

# Reload and verify
loaded = load_from_disk(CONFIG["output_dir"] + "/spider_formatted")

print("=== Verification ===")
print(f"Train samples     : {len(loaded['train'])}")
print(f"Validation samples: {len(loaded['validation'])}")
print(f"Columns           : {loaded['train'].column_names}")

print("\n--- Sample from saved dataset ---")
sample = loaded['train'][42]
print(f"Question : {sample['question']}")
print(f"Query    : {sample['query']}")
print(f"DB       : {sample['db_id']}")
print("\nPhase 1 Complete! Ready for Phase 2 - Baseline Evaluation")
